#  1. Import libraries

In [11]:
# 1. Import libraries
import os
import fitz  # PyMuPDF to read PDFs
from dotenv import load_dotenv
import google.generativeai as genai

 # 2. Configure API Key

In [12]:
# Load variables from .env file
load_dotenv()

# Assign Google API Key
api_key = os.getenv("GOOGLE_API_KEY")
genai.configure(api_key=api_key)


In [13]:
# List available models
for model in genai.list_models():
    print(model.name)

models/chat-bison-001
models/text-bison-001
models/embedding-gecko-001
models/gemini-1.0-pro-vision-latest
models/gemini-pro-vision
models/gemini-1.5-pro-latest
models/gemini-1.5-pro-001
models/gemini-1.5-pro-002
models/gemini-1.5-pro
models/gemini-1.5-flash-latest
models/gemini-1.5-flash-001
models/gemini-1.5-flash-001-tuning
models/gemini-1.5-flash
models/gemini-1.5-flash-002
models/gemini-1.5-flash-8b
models/gemini-1.5-flash-8b-001
models/gemini-1.5-flash-8b-latest
models/gemini-1.5-flash-8b-exp-0827
models/gemini-1.5-flash-8b-exp-0924
models/gemini-2.5-pro-exp-03-25
models/gemini-2.5-pro-preview-03-25
models/gemini-2.5-flash-preview-04-17
models/gemini-2.0-flash-exp
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.0-flash-lite-preview-02-05
models/gemini-2.0-flash-lite-preview
models/gemini-2.0-pro-exp
models/gemini-2.0-pro-exp-02-05
models/gemini-exp-1206
models/gemini-2.0-flash-thinking-exp-01-21
mod

# 3. Function to extract text from PDF

In [23]:
# 3. Function to extract text from PDF
def extract_text_from_pdf(pdf_path):
    doc = fitz.open(pdf_path)
    text = ""
    for page in doc:
        text += page.get_text()
    doc.close()
    return text

# 4.Prompt multilíngue 

In [20]:
# New prompt to analyse a resume in British English
PROMPT_ANALYSIS_EN = """
You are a Human Resources specialist focused on hiring Data Scientists.
Carefully review the following CV and respond with:

1. Score from 0 to 10 for:
   - Clarity
   - Impact
   - Suitability for Data Science roles

2. Identify key strengths.

3. Identify main weaknesses.

4. Provide concrete suggestions for improvement.

5. Is the CV ready for international job applications?

Here is the CV:
{text}
"""

PROMPT_ANALYSIS_PT = """
Você é um especialista em Recursos Humanos com foco em contratar Cientistas de Dados.
Revise cuidadosamente o seguinte currículo e responda com:

1. Dê uma nota de 0 a 10 para:
   - Clareza
   - Impacto
   - Adequação para vagas de Ciência de Dados

2. Pontos fortes principais.

3. Principais pontos fracos.

4. Sugestões concretas de melhoria.

5. O currículo está pronto para aplicações internacionais?

Aqui está o CV:
{text}
"""


# 4.1. # Initialize the model 

In [21]:
# Initialize the model (use another model if you want)
model = genai.GenerativeModel(model_name="models/gemini-1.5-flash")

def analyze_resume(text, lang="en"):
    prompt = PROMPT_ANALYSIS_EN.format(text=text) if lang == "en" else PROMPT_ANALYSIS_PT.format(text=text)
    response = model.generate_content(prompt)
    return response.text

# 5. Example of use: read and analyse a PDF

In [22]:
# Path of your PDF
pdf_path = "Profile.pdf"

# Extraction and analysis
pdf_text = extract_text_from_pdf(pdf_path)
analysis = analyze_resume(pdf_text)
# Em inglês
english_review = analyze_resume(pdf_text, lang="en")
print("🔎 English Review:\n", english_review)

# Em português
portuguese_review = analyze_resume(pdf_text, lang="pt")
print("🔎 Revisão em Português:\n", portuguese_review)


🔎 English Review:
 ## CV Review: Joao Martire

**1. Scoring:**

* **Clarity:** 6/10  (Information is present but disorganized and repetitive.  Sections could be streamlined.)
* **Impact:** 4/10 (Quantifiable achievements are mentioned but not always effectively presented or linked to data science skills.)
* **Suitability for Data Science roles:** 6/10 (Possesses relevant skills and projects, but the CV needs significant restructuring to highlight these effectively.)


**2. Key Strengths:**

* **Relevant Projects:** Joao has undertaken several projects demonstrating practical application of machine learning algorithms (clustering, ranking, regression) and resulting in quantifiable positive outcomes (revenue increase, efficiency improvements).  The portfolio link is a plus.
* **Technical Skills:** Possesses a solid foundation in programming languages (Python, SQL), databases (MySQL, Postgres, SQL Server, Snowflake), cloud computing (AWS), and machine learning techniques.
* **Bilingualism